# Controlled MobilePIXOR backbone ablations (Google Colab)

B0 is the MobilePIXOR reference; B1_C2PSA adds post-C5 C2PSA. C4_LSK and C4_LITEMLA each add one independent post-C4 refinement, feeding both C5 and the FPN. Presets share augmentation, loss, and training settings. Encoding, SG-FPN, and C5 overrides support controlled ablations. Each run is locked to its branch, commit, and resolved config before resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import math
import os
import torch

# Push the implementation/configs to this branch before starting Colab.
BRANCH = "decoupled-c4-c5-routing"
VARIANT = "C4_LITEMLA_C5_C2PSA_DECOUPLED"
CONFIG_OVERRIDE = None  # e.g. "configs/kitti/backbone_branch/my_proposal.json"
BEV_ENCODING_OVERRIDE = None  # None keeps config; or "binary_slices" / "rich8"
SCALE_GATED_FPN_OVERRIDE = None  # None keeps config; or False / True
C5_ATTENTION_OVERRIDE = None  # None keeps config; or "none" / "c2psa"
C4_ATTENTION_ROUTE_OVERRIDE = None  # None keeps config; or "shared" / "lateral_only"
RUN_NAME = None  # None creates a stable, resume-friendly name
SEED = 42
PRECISION = "auto"  # auto chooses BF16 when supported, otherwise FP16
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
TARGET_BACKEND = "numba"  # parity-tested; use python for the eager reference
COMPILE_MODEL = False  # Opt in only after timing a complete warm run
COMPILE_MODEL_ARGUMENT = "--compile-model" if COMPILE_MODEL else ""
RUNTIME_PROFILE = f"{TARGET_BACKEND}_{'compile' if COMPILE_MODEL else 'eager'}"

EPOCHS = 100  # Keep identical across compared runs
NUM_WORKERS = max(0, min(6, (os.cpu_count() or 1) - 1))
RUN_SMOKE_TEST = True
SMOKE_TRAIN_BATCHES = 8
SMOKE_VAL_BATCHES = 4
RUN_EVALUATION = True
ALLOW_LEGACY_RESUME = False  # True only for runs created before this notebook

REPOSITORY_URL = "https://github.com/danhyoyo/Lidar.git"
REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/lidar_training_artifacts")

drive.mount("/content/drive")
%cd /content
!test -d Lidar || git clone "{REPOSITORY_URL}" Lidar
if _exit_code:
    raise RuntimeError("Không clone được repository.")
%cd /content/Lidar
!git fetch --prune origin "+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không fetch được branch: {BRANCH}")
!git checkout --detach "origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không checkout được branch: {BRANCH}")
COMMIT = !git rev-parse HEAD
COMMIT = COMMIT[0]
print(f"Branch: {BRANCH}\nCommit: {COMMIT}")

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Runtime > Change runtime type trước khi train.")
if PRECISION == "auto":
    PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
elif PRECISION == "bf16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU này không hỗ trợ BF16; đổi PRECISION thành auto, fp16 hoặc fp32.")
print(PRECISION)
print(f"PyTorch: {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}")

## Dependencies and clean experiment selector

`CONFIG_OVERRIDE` accepts a repository-relative JSON path. `model.c4_attention` selects `none`, `lsk`, or `litemla`; `model.c4_attention_route` selects `shared` or `lateral_only`; `model.scale_gated_fpn` controls SG-FPN. C5 attention, routing, FPN, and BEV encoding are independent. Optional overrides above write a separate resolved config, used by architecture inspection, training, and evaluation. Repository presets are not overwritten. Run names include all switches and a config hash.

In [ ]:
%cd /content/Lidar
%pip install -q "numba>=0.59" shapely onnx tensorboard tqdm

VARIANT_CONFIGS = {
    "B0": "configs/kitti/backbone_branch/kitti_mobilepixor_baseline.json",
    "B1_C2PSA": "configs/kitti/backbone_branch/kitti_mobilepixor_c2psa.json",
    "C4_LSK": "configs/kitti/backbone_branch/kitti_mobilepixor_c4_lsk.json",
    "C4_LITEMLA": "configs/kitti/backbone_branch/kitti_mobilepixor_c4_litemla.json",
    "RICH8_SGFPN_CONTROL": "configs/kitti/backbone_branch/kitti_mobilepixor_rich8_sgfpn_control.json",
    "C4_LITEMLA_LATERAL_ONLY": "configs/kitti/backbone_branch/kitti_mobilepixor_c4_litemla_lateral_only.json",
    "C4_LITEMLA_C5_C2PSA_SHARED": "configs/kitti/backbone_branch/kitti_mobilepixor_c4_litemla_c5_c2psa_shared.json",
    "C4_LITEMLA_C5_C2PSA_DECOUPLED": "configs/kitti/backbone_branch/kitti_mobilepixor_c4_litemla_c5_c2psa_decoupled.json",
}
if CONFIG_OVERRIDE is None and VARIANT not in VARIANT_CONFIGS:
    raise ValueError(f"Unknown VARIANT={VARIANT!r}; use CONFIG_OVERRIDE for a new proposal.")
CONFIG_RELATIVE = CONFIG_OVERRIDE or VARIANT_CONFIGS[VARIANT]
CONFIG = (REPO_DIR / CONFIG_RELATIVE).resolve()
if REPO_DIR not in CONFIG.parents or not CONFIG.is_file():
    raise FileNotFoundError(f"Config unavailable on {BRANCH}: {CONFIG_RELATIVE}")
import sys
pipeline_dir = str((REPO_DIR / "tools/kitti_training_pipeline").resolve())
if pipeline_dir not in sys.path:
    sys.path.insert(0, pipeline_dir)
from common import read_json, write_json
from ablation import resolve_ablation_config, ablation_label, config_digest
resolved_config = resolve_ablation_config(
    read_json(CONFIG), bev_encoding=BEV_ENCODING_OVERRIDE,
    scale_gated_fpn=SCALE_GATED_FPN_OVERRIDE, c5_attention=C5_ATTENTION_OVERRIDE,
    c4_attention_route=C4_ATTENTION_ROUTE_OVERRIDE,
)
# Make the hashed/saved config describe the training command exactly.
resolved_config["seed"] = SEED
resolved_config["train"]["epochs"] = EPOCHS
resolved_config["train"]["precision"] = PRECISION
resolved_config["train"]["physical_batch_size"] = PHYSICAL_BATCH_SIZE
resolved_config["train"]["accumulation_steps"] = ACCUMULATION_STEPS
CONFIG_DIGEST = config_digest(resolved_config)
EXPERIMENT_LABEL = ablation_label(resolved_config)
CONFIG = REPO_DIR / "artifacts/notebook_configs" / f"{Path(CONFIG_RELATIVE).stem}_{CONFIG_DIGEST[:12]}.json"
write_json(CONFIG, resolved_config)
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS
RUN_NAME = RUN_NAME or f"{EXPERIMENT_LABEL}_{CONFIG_DIGEST[:8]}_seed{SEED}_eb{EFFECTIVE_BATCH_SIZE}_{RUNTIME_PROFILE}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Preset: {VARIANT}\nSource config: {CONFIG_RELATIVE}\nResolved config: {CONFIG}\nSwitches: {EXPERIMENT_LABEL}\nRun: {RUN_NAME}\nRuntime: {RUNTIME_PROFILE}")

## Inspect architecture and baseline delta

This builds both models on CPU, prints the selected architecture, and reports structural and trainable-parameter differences from B0 before any training starts.

In [ ]:
import gc
import sys

pipeline_dir = str((REPO_DIR / "tools/kitti_training_pipeline").resolve())
if pipeline_dir not in sys.path:
    sys.path.insert(0, pipeline_dir)

from common import build_model, configure_detector_imports, input_shape, read_json

configure_detector_imports(REPO_DIR / "detector")
baseline_config = read_json(REPO_DIR / "configs/kitti/backbone_branch/kitti_mobilepixor_baseline.json")
baseline_config = resolve_ablation_config(baseline_config, bev_encoding="binary_slices", scale_gated_fpn=False, c5_attention="none")
baseline_config["model"]["c4_attention"] = "none"
selected_config = read_json(CONFIG)
baseline_model = build_model(baseline_config).cpu()
selected_model = build_model(selected_config).cpu()

print("Selected architecture:")
print(selected_model)

def trainable_parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def module_types(model):
    return {path: type(module).__name__ for path, module in model.named_modules() if path}

def top_level_only(paths, blocked_parents=()):
    paths = sorted(paths)
    return [
        path for path in paths
        if not any(
            path.startswith(parent + ".")
            for parent in (*paths, *blocked_parents)
            if parent != path
        )
    ]

baseline_modules = module_types(baseline_model)
selected_modules = module_types(selected_model)
shared_paths = baseline_modules.keys() & selected_modules.keys()
changed_paths = sorted(
    path for path in shared_paths
    if baseline_modules[path] != selected_modules[path]
)
added_paths = top_level_only(selected_modules.keys() - baseline_modules.keys(), changed_paths)
removed_paths = top_level_only(baseline_modules.keys() - selected_modules.keys(), changed_paths)
baseline_parameters = trainable_parameter_count(baseline_model)
selected_parameters = trainable_parameter_count(selected_model)

print("\nDifferences from baseline:")
print(f"- Input shape: {input_shape(baseline_config)} -> {input_shape(selected_config)}")
for field in ("c4_attention", "c4_attention_route", "c5_attention", "scale_gated_fpn", "header_use_bn", "header_act"):
    print(f"- {field}: {baseline_config['model'].get(field)} -> {selected_config['model'].get(field)}")
if not changed_paths and not added_paths and not removed_paths:
    print("- No structural module changes.")
for path in changed_paths:
    print(f"- Changed {path}: {baseline_modules[path]} -> {selected_modules[path]}")
for path in added_paths:
    print(f"- Added {path}: {selected_modules[path]}")
for path in removed_paths:
    print(f"- Removed {path}: {baseline_modules[path]}")
print(f"- Baseline trainable parameters: {baseline_parameters:,}")
print(f"- Selected trainable parameters: {selected_parameters:,}")
print(f"- Parameter delta: {selected_parameters - baseline_parameters:+,}")

del baseline_model, selected_model
gc.collect()

## Prepare KITTI

The cell is idempotent: complete folders are reused; incomplete folders are re-extracted and validated.

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted_dir = RAW_KITTI_ROOT / "training" / folder
    extracted_count = sum(1 for _ in extracted_dir.glob(pattern))

    if extracted_count != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Không tìm thấy archive: {archive}")
        archive_bytes = archive.stat().st_size
        !set -o pipefail; python3 -m tqdm --bytes --total {archive_bytes} --desc "Giải nén {folder}" < "{archive}" | tar --no-same-owner -xf - -C "{RAW_KITTI_ROOT}"
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")

    actual_count = sum(1 for _ in extracted_dir.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: {actual_count} file, cần {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)

if not dataset_ready:
    %cd /content/Lidar
    !python3 tools/kitti_training_pipeline/prepare_kitti.py \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --output-root "{PROCESSED_DATASET_DIR}" \
      --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" \
      --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" \
      --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" \
      --pointcloud-mode symlink \
      --overwrite
    if _exit_code:
        raise RuntimeError("prepare_kitti.py thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
assert pointcloud_count == label_count == 7481
assert (PROCESSED_DATASET_DIR / "train.txt").is_file()
assert (PROCESSED_DATASET_DIR / "val.txt").is_file()

print(f"Raw KITTI: {RAW_KITTI_ROOT}")
print(f"Processed: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count}")


## Verify the checked-out backbone implementations

Run this before the smoke/full training cells, so each branch proves its own test contract.

In [ ]:
%cd /content/Lidar
!MPLCONFIGDIR=/tmp/lidar-mpl python3 tests/test_mobile_bev.py --clean-backbone
if _exit_code:
    raise RuntimeError("Proposal tests failed; do not train this checkout.")
!python3 tests/test_c4_attention.py
if _exit_code:
    raise RuntimeError("C4 ablation tests failed; do not train this checkout.")

## Smoke test

This uses a separate run directory, so it never contaminates a resumable full run.

In [ ]:
SMOKE_RUN_NAME = f"{RUN_NAME}_smoke"
SMOKE_DIR = ARTIFACT_ROOT / SMOKE_RUN_NAME
SMOKE_METRICS = SMOKE_DIR / "metrics.jsonl"
SMOKE_CHECKPOINT = SMOKE_DIR / "checkpoints/last.pt"
if RUN_SMOKE_TEST:
    !python3 tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{SMOKE_RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs 1 --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --max-train-batches {SMOKE_TRAIN_BATCHES} --max-val-batches {SMOKE_VAL_BATCHES} --num-workers {NUM_WORKERS} --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT}
    if _exit_code:
        raise RuntimeError(f"Smoke test thất bại cho runtime {RUNTIME_PROFILE}.")
    if not SMOKE_CHECKPOINT.is_file() or not SMOKE_METRICS.is_file():
        raise RuntimeError("Smoke test không tạo đủ checkpoint/metrics.")
    smoke_rows = [line for line in SMOKE_METRICS.read_text(encoding="utf-8").splitlines() if line]
    smoke_row = json.loads(smoke_rows[-1]) if smoke_rows else None
    if not smoke_row or smoke_row["epoch"] != 1:
        raise RuntimeError("Smoke metrics thiếu epoch 1.")
    if smoke_row["optimizer_updates"] < 1:
        raise RuntimeError("Smoke test không thực hiện optimizer update.")
    if not math.isfinite(smoke_row["train_objective"]):
        raise RuntimeError("Smoke train objective không hữu hạn.")
    if not math.isfinite(smoke_row["validation"]["loss"]):
        raise RuntimeError("Smoke validation loss không hữu hạn.")
    print(f"Smoke PASS [{RUNTIME_PROFILE}]: {SMOKE_CHECKPOINT}")

## Full training / safe resume

The trainer writes `checkpoints/last.pt`, appends one row to `metrics.csv`, and updates TensorBoard in `tensorboard/` after every completed epoch. Training stays attached to this cell; if interrupted, rerun it to resume from the latest completed epoch.

In [ ]:
import hashlib
import re

RUN_DIR = ARTIFACT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
TRAIN_LOG = RUN_DIR / "train.log"
RUN_METADATA_PATH = RUN_DIR / "run.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_METADATA = {"branch": BRANCH, "commit": COMMIT, "variant": VARIANT, "config_relative": CONFIG_RELATIVE, "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest(), "seed": SEED, "precision": PRECISION, "physical_batch_size": PHYSICAL_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS, "target_backend": TARGET_BACKEND, "compile_model": COMPILE_MODEL, "num_workers": NUM_WORKERS}
if RUN_METADATA_PATH.is_file():
    if json.loads(RUN_METADATA_PATH.read_text(encoding="utf-8")) != RUN_METADATA:
        raise RuntimeError("Run metadata differs; choose a new RUN_NAME instead of resuming incompatible state.")
elif any(CHECKPOINT_DIR.glob("*.pt")) and not ALLOW_LEGACY_RESUME:
    raise RuntimeError("Existing checkpoints have no run.json. Set ALLOW_LEGACY_RESUME=True only after verifying compatibility.")
else:
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2, sort_keys=True) + "\n", encoding="utf-8")

def checkpoint_epoch(path):
    if not path.is_file():
        return -1
    state = torch.load(path, map_location="cpu", weights_only=False)
    return int(state.get("epoch", -1)) if isinstance(state, dict) else -1

last_checkpoint = CHECKPOINT_DIR / "last.pt"
epoch_checkpoints = [path for path in CHECKPOINT_DIR.glob("*epoch.pt") if re.fullmatch(r"\d+epoch\.pt", path.name)]
resume_checkpoint = last_checkpoint if last_checkpoint.is_file() else max(epoch_checkpoints, key=checkpoint_epoch, default=None)
resume_epoch = checkpoint_epoch(resume_checkpoint) if resume_checkpoint else 0
if resume_epoch >= EPOCHS:
    print(f"Training already reached epoch {resume_epoch}/{EPOCHS}.")
else:
    RESUME_ARGUMENT = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""
    if resume_checkpoint:
        print(f"Resuming epoch {resume_epoch}: {resume_checkpoint}")
    !set -o pipefail; python3 -u tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs {EPOCHS} --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --num-workers {NUM_WORKERS} --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT} {RESUME_ARGUMENT} 2>&1 | tee -a "{TRAIN_LOG}"
    if _exit_code:
        raise RuntimeError(f"Training failed with exit code {_exit_code}")

## Select checkpoint and evaluate

All variants use the trainer's minimum-validation-loss checkpoint and BEV AP evaluation.

In [ ]:
CHECKPOINT = RUN_DIR / "selected" / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)

if RUN_EVALUATION:
    split = REPO_DIR / "splits/kitti/val.txt"
    output = RUN_DIR / "evaluation_validation.json"
    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py --name "{RUN_NAME}_validation" --backend pytorch --model "{CHECKPOINT}" --config "{CONFIG}" --detector-root detector --kitti-root "{RAW_KITTI_ROOT}" --split "{split}" --output "{output}" --device cuda --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá validation thất bại.")
print(f"Checkpoint: {CHECKPOINT}")